hi

In [0]:
%fs
ls

In [0]:
%sh
ls

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F

def calculate_rolling_sales():
    # 1. Initialize Spark Session
    spark = SparkSession.builder.appName("30DayRollingSales").getOrCreate()

    # Sample Data
    data = [
        ("Store_1", "2023-01-01", 100),
        ("Store_1", "2023-01-15", 200),
        ("Store_1", "2023-01-20", 150),
        ("Store_1", "2023-02-05", 300), # This is > 30 days from Jan 1st
    ]
    df = spark.createDataFrame(data, ["store", "sales_date", "sales"])

    print("--- Original Data ---")
    df.show()

    # 2. Convert date string to DateType, then to timestamp (seconds)
    # 86400 is the number of seconds in a day (24 hours * 60 min * 60 sec)
    df = df.withColumn("sales_date", F.to_date(F.col("sales_date"))) \
           .withColumn("date_in_seconds", F.unix_timestamp(F.col("sales_date")))

    # 3. Define the 30-day Window
    days_30_in_seconds = 30 * 86400

    window_30_days = Window.partitionBy("store") \
                           .orderBy("date_in_seconds") \
                           .rangeBetween(-days_30_in_seconds, Window.currentRow)

    # 4. Calculate the 30-day rolling sum and average
    result_df = df.withColumn("30_day_rolling_sum", F.sum("sales").over(window_30_days)) \
                  .withColumn("30_day_rolling_avg", F.avg("sales").over(window_30_days))

    # Drop the temporary seconds column to clean up
    result_df = result_df.drop("date_in_seconds")

    print("--- Data with 30-Day Rolling Sales ---")
    result_df.show()

if __name__ == "__main__":
    calculate_rolling_sales()
